# Stage 05: Data Storage

Environment-driven CSV and Parquet storage with reusable utilities and validation.


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    candidate = PROJECT_ROOT / "homework" / "homework05"
    if candidate.exists():
        PROJECT_ROOT = candidate
load_dotenv(PROJECT_ROOT / ".env")
RAW_DIR = PROJECT_ROOT / os.getenv("DATA_DIR_RAW", "data/raw")
PROCESSED_DIR = PROJECT_ROOT / os.getenv("DATA_DIR_PROCESSED", "data/processed")
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.storage import read_df, validate_reloads, write_df
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)


Raw directory: homework/homework05/data/raw
Processed directory: homework/homework05/data/processed


## 1. Create a typed market DataFrame


In [2]:
records = [
("2026-08-20",236.00,237.50,233.03,233.69,4028353),
("2026-08-19",232.26,238.615,230.51,237.16,5342567),
("2026-08-18",229.73,234.4499,229.415,232.67,4650116),
("2026-08-17",231.59,233.65,227.40,228.85,7564317),
("2026-08-14",238.37,239.15,233.73,234.32,4523467)]
columns = ["date","open","high","low","close","volume"]
df = pd.DataFrame(records, columns=columns)
df["date"] = pd.to_datetime(df["date"])
df = df.astype({"open":"float64","high":"float64","low":"float64","close":"float64","volume":"int64"})
df


        date    open      high      low   close   volume
0 2026-08-20  236.00  237.5000  233.030  233.69  4028353
1 2026-08-19  232.26  238.6150  230.510  237.16  5342567
2 2026-08-18  229.73  234.4499  229.415  232.67  4650116
3 2026-08-17  231.59  233.6500  227.400  228.85  7564317
4 2026-08-14  238.37  239.1500  233.730  234.32  4523467

## 2. Save CSV and Parquet


In [3]:
timestamp = "20260820-0000"
csv_path = RAW_DIR / f"ibm_prices_{timestamp}.csv"
parquet_path = PROCESSED_DIR / f"ibm_prices_{timestamp}.parquet"
write_df(df, csv_path)
write_df(df, parquet_path)
print("Saved CSV:", csv_path)
print("Saved Parquet:", parquet_path)


Saved CSV: homework/homework05/data/raw/ibm_prices_20260820-0000.csv
Saved Parquet: homework/homework05/data/processed/ibm_prices_20260820-0000.parquet


## 3. Reload and validate


In [4]:
csv_reloaded = read_df(csv_path)
parquet_reloaded = read_df(parquet_path)
checks = validate_reloads(df, csv_reloaded, parquet_reloaded)
for name, passed in checks.items():
    print(f"{name}: {passed}")
assert all(checks.values()), "One or more storage validations failed"


csv_shape_matches: True
parquet_shape_matches: True
csv_columns_match: True
parquet_columns_match: True
csv_date_is_datetime: True
parquet_date_is_datetime: True
csv_close_is_float: True
parquet_close_is_float: True
csv_volume_is_integer: True
parquet_volume_is_integer: True


## Storage choices

CSV is stored in data/raw because it is readable and portable. Parquet is stored in data/processed because it preserves types and supports efficient columnar analytics. Paths come from .env. The storage utilities route by suffix, create directories, detect missing files, and explain how to install a missing Parquet engine.
